### Web Scraping : Involves programmatically extracting data from websites
- When we click on a link it sends a request and the response it gets is HTML, JS or CSS files
- We try to render these file only and get data from it
- Mainly we will use HTML files to get content using "requests" and "beautiful soup"
- Generally, API > WebScraping


In [1]:
import requests
URL = "https://www.scrapethissite.com/pages/simple/"
response = requests.get(URL)

In [2]:
# printing the stored html page

if response.status_code == 200: # 200 means all okay
     # print(response.text) # .text prints out all the data of website as string
     # print(response.content) # .content prints raw content in bytes
       print(response.headers) # .headers prints metadata like content type, datetime etc



# saving the scraped data in a local file

with open("Scraped_data/Data1.html","w") as f:
    f.write(response.text)

{'Date': 'Sat, 30 May 2026 05:52:34 GMT', 'Content-Type': 'text/html; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Nel': '{"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}', 'Report-To': '{"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=2n%2FkS%2FaAJbOz8Q7u2Gv94Zc4iBM1ogYoUs0I7TC%2BlCI%3D\\u0026sid=67ff5de4-ad2b-4112-9289-cf96be89efed\\u0026ts=1780120353"}],"max_age":3600}', 'Reporting-Endpoints': 'heroku-nel="https://nel.heroku.com/reports?s=2n%2FkS%2FaAJbOz8Q7u2Gv94Zc4iBM1ogYoUs0I7TC%2BlCI%3D&sid=67ff5de4-ad2b-4112-9289-cf96be89efed&ts=1780120353"', 'Server': 'cloudflare', 'Via': '1.1 heroku-router', 'Cf-Cache-Status': 'DYNAMIC', 'Content-Encoding': 'zstd', 'CF-RAY': 'a03b79308ddb203d-SIN', 'alt-svc': 'h3=":443"; ma=86400'}


# Working With Beautiful Soup (bs4)
- after downloading and storing the html data we parse it
- parse : understand and extract the structrure and content of the html file


In [5]:
pip install lxml # used for fast parsing xml and html files
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [29]:
from bs4 import BeautifulSoup

with open("Scraped_data/Data1.html","r") as f:
    html_content = f.read()

# soup is an object of the BeautifulSoup class
soup = BeautifulSoup(html_content,"lxml")

# now we need country name and its population for our ML model
# the main issue is extra data in the file and how to convert html data to normal table type format
# we generally use FIND methods of beauutifulsoup class

soup.find("h3") #1. finds the first occurence of the tag
all_h3 = soup.find_all("h3") #2. finds all occurences of the tag

type(all_h3) # bs4.element.ResultSet behaves just like a list so we can run a loop

all_countries_info = [] # empty list to store the values

for h3 in all_h3:
    name = (h3.get_text(strip=True)) # prints info of all h3's, but we need content inside h3 tag
                                     # strip cancels the spaces

    pop = (h3.find_next("div").select_one("span.country-population").get_text()) 
    #3. find_next finds occurence of the sibling tag, using to find pop
    # .select is use to access any css element and returns a list, with . you can access any class
    # .select_one returns the first item, and we use this 

    all_countries_info.append([name, pop])

In [37]:
# now we have the data, just convert into df and do EDA

import pandas as pd

df = pd.DataFrame(all_countries_info, columns = ["Country", "Population"])
df

# now we can store this cleaned data in a new csv file

df.to_csv("Cleaned_data/Population_info.csv", index=False)